<div style="text-align:center;">

<h1><b>UNIVERSIDAD NACIONAL AGRARIA LA MOLINA</b></h1>

<h2><b>DEPARTAMENTO ACADÉMICO DE ESTADÍSTICA E INFORMÁTICA</b></h2>

<img src="https://seeklogo.com/images/U/universidad-nacional-agraria-la-molina-logo-5BF0B8D973-seeklogo.com.png"
     alt="La Molina Perú"
     width="170"
     height="170"
     style="display:block; margin-left:auto; margin-right:auto;">

<h3><b>ANÁLISIS DESCRIPTIVO E INFERENCIAL DE LA EJECUCIÓN DEL GASTO PÚBLICO EN EL PERÚ DURANTE SEPTIEMBRE DE 2024 CON DATOS ABIERTOS DEL MEF</b></h3>

</div>
<h3 align="center">
  <b>ANÁLISIS DESCRIPTIVO E INFERENCIAL DE LA EJECUCIÓN DEL GASTO PÚBLICO EN EL PERÚ DURANTE SEPTIEMBRE DE 2024 CON DATOS ABIERTOS DEL MEF</b>
</h3>

<hr>

<h3> <b>Integrantes del equipo:</b> </h3>

<ul>
  <li>Cárdenas Panduro, Ricardo Gabriel (Rick2425) (20241376)</li>
  <li>Almonacid Quispe, Jimmy Salomón (patatita1theoriginal) (20241374)</li>
  <li>Tuppia Paitan, Joaquin Francisco (JTPXD) (20241405)</li>
  <li>Ortiz Huamani, Ricardo Fidel (ricardofortizh) (20240724)</li>
</ul>

# **1. EXTRACCIÓN DE DATOS MEDIANTE LA API DEL MEF**
Su función principal es conectarse a la API de Datos Abiertos del Ministerio de Economía y Finanzas, descargar registros del gasto público en bloques y guardarlos en un archivo CSV llamado:
*gasto_publico_raw_2024.csv*

¿Que hace este bloque?
1. Configura los parámetros de extracción:
   - ID del recurso de la API.
   - Nombre del archivo de salida.
   - Cantidad máxima de registros por bloque.
   - Meta de 100 000 registros.
   - Tiempo de espera y reintentos.

2. Crea una interfaz de carga en consola:
   - Muestra una barra de progreso visual antes de iniciar la extracción.

3. Se conecta a la API del MEF:
   - Usa la librería `requests`.
   - Envía solicitudes HTTP tipo GET.
   - Recibe la respuesta en formato JSON.
   - Valida errores de conexión o respuesta.

4. Extrae los registros:
   - Busca los datos dentro del JSON recibido.
   - Soporta dos estructuras posibles: `records` o `result["records"]`.

5. Guarda los datos:
   - Convierte los registros a un DataFrame de pandas.
   - Los guarda progresivamente en un archivo CSV.
   - Si ya existe un archivo anterior, lo elimina para empezar una nueva descarga.

6. Controla el avance:
   - Descarga datos por bloques usando `offset`.
   - Muestra cuántos registros se guardaron.
   - Detiene el proceso cuando llega a 100 000 registros o cuando ya no hay más datos.

In [ ]:
import requests
import pandas as pd
import time
import os
import sys
from pathlib import Path
from dataclasses import dataclass

# ============================================================
# CONFIGURACIÓN GENERAL DE EXTRACCIÓN
# ============================================================

@dataclass
class MEFConfig:
    resource_id: str
    nombre_archivo: str = "gasto_publico_raw_2024.csv"
    limite_por_bloque: int = 32000
    meta_registros: int = 100000
    timeout: int = 120
    max_reintentos: int = 3
    duracion_carga: int = 6

    url_base: str = "https://api.datosabiertos.mef.gob.pe/DatosAbiertos/v1/datastore_search"

# Debido a limitaciones de la API al extraer la data dentro de nuestro repositorio, la
# capacidad de extracción se redujo, por lo que este código sera meramente representativo.

# ============================================================
# INTERFAZ DE CARGA EN CONSOLA
# ============================================================

class ConsoleLoader:
    def __init__(self, duracion: int = 6, ancho_barra: int = 30):
        self.duracion = duracion
        self.ancho_barra = ancho_barra

    def mostrar(self, mensaje: str = "Preparando extracción"):
        print(mensaje)

        for porcentaje in range(101):
            progreso = int((porcentaje / 100) * self.ancho_barra)
            barra = "█" * progreso + "░" * (self.ancho_barra - progreso)

            sys.stdout.write(f"\r[{barra}] {porcentaje:3d}%")
            sys.stdout.flush()

            time.sleep(self.duracion / 100)

        print("\nCarga completada.\n")


# ============================================================
# CLIENTE API MEF
# ============================================================

class MEFAPIClient:
    def __init__(self, config: MEFConfig):
        self.config = config
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json",
            "Connection": "keep-alive"
        })

    def solicitar_bloque(self, offset: int) -> dict:
        params = {
            "resource_id": self.config.resource_id,
            "limit": self.config.limite_por_bloque,
            "offset": offset
        }

        for intento in range(1, self.config.max_reintentos + 1):
            try:
                respuesta = self.session.get(
                    self.config.url_base,
                    params=params,
                    timeout=self.config.timeout
                )

                respuesta.raise_for_status()
                return respuesta.json()

            except requests.exceptions.RequestException as error:
                print(f"\nError HTTP en intento {intento}: {error}")

                if intento == self.config.max_reintentos:
                    raise

                espera = intento * 3
                print(f"Reintentando en {espera} segundos...")
                time.sleep(espera)

    @staticmethod
    def extraer_registros(data_json: dict) -> list:
        registros = data_json.get("records", [])

        if not registros:
            registros = data_json.get("result", {}).get("records", [])

        return registros


# ============================================================
# ESCRITOR CSV
# ============================================================

class CSVRawWriter:
    def __init__(self, ruta_salida: str):
        self.ruta_salida = ruta_salida
        self.es_primer_bloque = True

    def reiniciar_archivo(self):
        if os.path.exists(self.ruta_salida):
            os.remove(self.ruta_salida)
            print("Archivo previo eliminado. Iniciando nueva descarga.")

    def guardar_bloque(self, registros: list):
        if not registros:
            return

        df = pd.DataFrame(registros)

        df.to_csv(
            self.ruta_salida,
            mode="a",
            index=False,
            header=self.es_primer_bloque,
            encoding="utf-8-sig"
        )

        self.es_primer_bloque = False


# ============================================================
# EXTRACTOR PRINCIPAL
# ============================================================

class MEFExtractor:
    def __init__(self, config: MEFConfig, cliente_api: MEFAPIClient, escritor_csv: CSVRawWriter):
        self.config = config
        self.cliente_api = cliente_api
        self.escritor_csv = escritor_csv
        self.total_guardado = 0

    def calcular_registros_a_guardar(self, registros: list) -> list:
        registros_restantes = self.config.meta_registros - self.total_guardado

        if len(registros) > registros_restantes:
            return registros[:registros_restantes]

        return registros

    def mostrar_progreso_real(self):
        porcentaje = int((self.total_guardado / self.config.meta_registros) * 100)
        porcentaje = min(porcentaje, 100)

        ancho_barra = 30
        progreso = int((porcentaje / 100) * ancho_barra)
        barra = "█" * progreso + "░" * (ancho_barra - progreso)

        print(f"Progreso real: [{barra}] {porcentaje}%")

    def ejecutar(self):
        offset = 0
        pagina_actual = 1

        self.escritor_csv.reiniciar_archivo()

        loader = ConsoleLoader(duracion=self.config.duracion_carga)
        loader.mostrar("Cargando extractor MEF")

        print("Iniciando extracción incremental desde la API del MEF...")

        while self.total_guardado < self.config.meta_registros:
            print(f"\nConsultando bloque {pagina_actual} | Offset: {offset}")

            try:
                data_json = self.cliente_api.solicitar_bloque(offset)
                registros = self.cliente_api.extraer_registros(data_json)

                if not registros:
                    print("No hay más registros disponibles en el servidor.")
                    break

                registros = self.calcular_registros_a_guardar(registros)

                self.escritor_csv.guardar_bloque(registros)

                self.total_guardado += len(registros)

                print(f"Guardados en este bloque: {len(registros)}")
                print(f"Total acumulado: {self.total_guardado}")

                self.mostrar_progreso_real()

                if self.total_guardado >= self.config.meta_registros:
                    print("Meta de 100,000 registros alcanzada.")
                    break

                if len(registros) < self.config.limite_por_bloque:
                    print("Último bloque recibido. No hay más datos.")
                    break

                offset += self.config.limite_por_bloque
                pagina_actual += 1

            except Exception as error:
                print(f"Error crítico en el bloque {pagina_actual}: {error}")
                break

        print("\nProceso finalizado.")
        print(f"Archivo generado: {self.escritor_csv.ruta_salida}")


# ============================================================
# EJECUCIÓN
# ============================================================

if __name__ == "__main__":

    ID_RECURSO = "a50cf1dc-1655-446d-95a3-de6d5351dc8c"

    try:
        carpeta_script = Path(__file__).resolve().parent
    except NameError:
        carpeta_script = Path.cwd()

    ruta_archivo = carpeta_script / "gasto_publico_raw_2024.csv"

    config = MEFConfig(
        resource_id=ID_RECURSO,
        nombre_archivo="gasto_publico_raw_2024.csv",
        limite_por_bloque=32000,
        meta_registros=100000,
        duracion_carga=6
    )

    cliente_api = MEFAPIClient(config)
    escritor_csv = CSVRawWriter(str(ruta_archivo))
    extractor = MEFExtractor(config, cliente_api, escritor_csv)

    extractor.ejecutar()


# **2. Limpieza, transformación y validación de datos**

En esta segunda fase del proyecto se realiza el proceso de limpieza y normalización de la base cruda obtenida desde la API de Datos Abiertos del Ministerio de Economía y Finanzas. Esta etapa fue desarrollada en **R**, por lo que dentro de Google Colab se presenta como una fase documentada, ya que Colab trabaja principalmente con celdas de Python y no ejecuta directamente scripts de R sin una configuración adicional.

El objetivo principal de este bloque es transformar el archivo crudo `gasto_publico_raw_2024.csv` en una base limpia y lista para el análisis estadístico e inferencial. Para ello, se seleccionan las variables necesarias, se corrigen formatos, se filtra el periodo de estudio y se genera una variable derivada llamada `tasa_giro`.

### Entrada y salida del proceso

| Elemento | Descripción |
|---|---|
| Archivo de entrada | `gasto_publico_raw_2024.csv` |
| Archivo de salida | `gasto_publico_limpio_inferencia.csv` |
| Archivo resumen | `resumen_limpieza.csv` |
| Año de estudio | 2024 |
| Mes de estudio | Septiembre |
| Software usado | R / RStudio |
| Paquete principal | `data.table` |

### Procedimiento realizado

El script inicia limpiando el entorno de trabajo de R y configurando la carpeta donde se encuentra el archivo de datos. Luego verifica que el archivo `gasto_publico_raw_2024.csv` exista en la carpeta actual. Si el archivo no se encuentra, el programa muestra los archivos CSV disponibles y detiene la ejecución.

Después, se definen funciones auxiliares para normalizar nombres de columnas, limpiar variables de texto y convertir valores numéricos de forma segura. Estas funciones permiten corregir problemas comunes en bases públicas, como espacios adicionales, tildes, símbolos monetarios, porcentajes, comas, puntos y valores vacíos.

### Limpieza de variables

En esta fase se seleccionan únicamente las columnas necesarias para el análisis:

| Variable | Uso dentro del análisis |
|---|---|
| `ano_eje` | Identifica el año de ejecución presupuestal. |
| `mes_eje` | Identifica el mes de ejecución presupuestal. |
| `nivel_gobierno_nombre` | Permite comparar el gasto según nivel de gobierno. |
| `sector_nombre` | Permite analizar el gasto por sector institucional. |
| `departamento_ejecutora_nombre` | Permite comparar el gasto por departamento ejecutor. |
| `funcion_nombre` | Permite identificar las funciones del gasto público. |
| `fuente_financiamiento_nombre` | Permite analizar el origen del financiamiento. |
| `categoria_gasto_nombre` | Permite comparar gasto corriente y gasto de capital. |
| `monto_devengado` | Variable principal del análisis. |
| `monto_girado` | Variable auxiliar usada para calcular la tasa de giro. |

La columna `monto_girado` se utiliza solo de manera temporal para construir la variable `tasa_giro`. Luego, se elimina de la base final para cumplir con el criterio de trabajar con un máximo de 10 variables.

### Filtro temporal

El análisis se delimita al mes de septiembre de 2024. Por ello, el script filtra los registros donde:

```txt
ano_eje = 2024
mes_eje = 9
````

Este filtro permite reducir la base de datos y concentrar el análisis en un periodo específico, facilitando la comparación entre sectores, regiones, funciones y categorías de gasto.

### Validación y depuración de registros

Luego del filtro temporal, el script elimina registros que puedan afectar el análisis. En particular:

* Elimina filas sin `monto_devengado`.
* Elimina registros con montos devengados negativos.
* Reemplaza valores vacíos de `monto_girado` por cero.
* Elimina registros con montos girados negativos.
* Controla valores faltantes en variables categóricas clave.
* Elimina duplicados exactos en la base final.

Este procedimiento permite obtener una base más consistente para el análisis descriptivo e inferencial.

### Variable derivada: tasa de giro

A partir de las variables `monto_girado` y `monto_devengado`, se calcula la variable:

```txt
tasa_giro = (monto_girado / monto_devengado) × 100
```

Esta variable representa el porcentaje del monto devengado que llegó a girarse. Es útil porque permite complementar el análisis del gasto público, ya que el monto devengado indica una obligación reconocida, mientras que la tasa de giro aproxima el avance financiero posterior.

Para evitar distorsiones, el script asigna como valores faltantes aquellas tasas menores que 0 o mayores que 150, ya que podrían corresponder a registros administrativos atípicos.

### Base final generada

La base final queda compuesta por 10 variables:

```txt
ano_eje
mes_eje
nivel_gobierno_nombre
sector_nombre
departamento_ejecutora_nombre
funcion_nombre
fuente_financiamiento_nombre
categoria_gasto_nombre
monto_devengado
tasa_giro
```

Esta base se guarda con el nombre:

```txt
gasto_publico_limpio_inferencia.csv
```

Además, se genera un archivo resumen llamado:

```txt
resumen_limpieza.csv
```

Este archivo resume la cantidad de filas iniciales, columnas iniciales, año de estudio, mes de estudio, filas posteriores al filtro mensual, filas finales y columnas finales.

### Importancia de esta fase

La limpieza, transformación y validación de datos es una etapa fundamental del proyecto, porque asegura que el análisis posterior se realice sobre una base ordenada, coherente y delimitada temporalmente. Sin esta fase, los resultados obtenidos en pandas y en los gráficos podrían verse afectados por registros inconsistentes, valores faltantes, duplicados o variables mal formateadas.

En conclusión, este bloque prepara la base de datos necesaria para continuar con el análisis de indicadores por sector, región y nivel de gobierno.

# **3. Analisis con pandas e indicadores**
Este bloque corresponde a la fase de **análisis con Python**. Su función principal es tomar la base limpia `gasto_publico_limpio_inferencia.csv` y generar resúmenes del gasto público usando la librería `pandas`.

El código define una clase llamada `AnalizadorGasto`, la cual se encarga de cargar los datos, convertir las variables numéricas y calcular indicadores agrupados por sector, región y nivel de gobierno.

### ¿Qué hace este código?

1. **Carga la base limpia**

   El código lee el archivo CSV que fue generado en la etapa anterior de limpieza y normalización.

   ```txt
   gasto_publico_limpio_inferencia.csv

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

class AnalizadorGasto:
    def __init__(self, ruta_csv):
        print(f"Cargando datos desde: {ruta_csv}...")
        self.df = pd.read_csv(ruta_csv)

        # Convertir a numérico y llenar nulos con 0
        self.df['monto_devengado'] = pd.to_numeric(self.df['monto_devengado'], errors='coerce').fillna(0)
        self.df['tasa_giro'] = pd.to_numeric(self.df['tasa_giro'], errors='coerce').fillna(0)

        print(f"Base cargada con {self.df.shape[0]} filas y {self.df.shape[1]} columnas.\n")

    def gasto_por_sector(self):
        """Agrupa el gasto devengado y promedia la tasa de giro por sector."""
        resumen = self.df.groupby('sector_nombre', as_index=False).agg(
            gasto_total_devengado=('monto_devengado', 'sum'),
            tasa_giro_promedio=('tasa_giro', 'mean')
        )
        resumen['tasa_giro_promedio'] = resumen['tasa_giro_promedio'].round(2)
        return resumen.sort_values(by='gasto_total_devengado', ascending=False)

    def gasto_por_region(self):
        """Agrupa el gasto devengado y promedia la tasa de giro por región."""
        resumen = self.df.groupby('departamento_ejecutora_nombre', as_index=False).agg(
            gasto_total_devengado=('monto_devengado', 'sum'),
            tasa_giro_promedio=('tasa_giro', 'mean')
        )
        resumen['tasa_giro_promedio'] = resumen['tasa_giro_promedio'].round(2)
        return resumen.sort_values(by='gasto_total_devengado', ascending=False)

    def gasto_por_nivel_gobierno(self):
        """Analiza la distribución del gasto según el nivel de gobierno."""
        resumen = self.df.groupby('nivel_gobierno_nombre', as_index=False).agg(
            gasto_total_devengado=('monto_devengado', 'sum'),
            tasa_giro_promedio=('tasa_giro', 'mean'),
            cantidad_proyectos=('monto_devengado', 'count')
        )
        resumen['tasa_giro_promedio'] = resumen['tasa_giro_promedio'].round(2)
        return resumen.sort_values(by='gasto_total_devengado', ascending=False)

    def ranking_top(self, df_agrupado, columna_orden='gasto_total_devengado', top=10):
        """Extrae los primeros registros según la columna de orden especificada."""
        return df_agrupado.nlargest(top, columna_orden)

    def guardar_tabla(self, df_tabla, ruta_salida):
        """Guarda el DataFrame en un archivo CSV."""
        ruta = Path(ruta_salida)
        ruta.parent.mkdir(parents=True, exist_ok=True)
        df_tabla.to_csv(ruta, index=False, encoding='utf-8-sig')
        print(f"Archivo generado: {ruta}")

# ==========================================
# EJECUCIÓN DEL SCRIPT
# ==========================================
if __name__ == '__main__':
    # 1. Instanciar la clase con el archivo de datos limpio
    analizador = AnalizadorGasto(r'C:\Users\tuppi\Documents\LP2 PRACTICA\gasto_publico_limpio_inferencia.csv')

    # 2. Generar los DataFrames con los indicadores
    df_sector = analizador.gasto_por_sector()
    df_region = analizador.gasto_por_region()
    df_nivel = analizador.gasto_por_nivel_gobierno()

    # 3. Generar Rankings
    top_sectores = analizador.ranking_top(df_sector)

    # 4. Guardar los resultados en la carpeta processed
    print("Guardando archivos...")
    analizador.guardar_tabla(df_sector, 'data/processed/resumen_sector.csv')
    analizador.guardar_tabla(df_region, 'data/processed/resumen_region.csv')
    analizador.guardar_tabla(df_nivel, 'data/processed/resumen_nivel_gobierno.csv')
    analizador.guardar_tabla(top_sectores, 'data/processed/ranking_sectores.csv')

    print("\nFase 3 completada")

## 4. Visualización de resultados

Este bloque corresponde a la fase de **visualización de resultados** del proyecto. Su función principal es tomar la base limpia `gasto_publico_limpio_inferencia.csv` y generar gráficos que permitan interpretar de forma visual la ejecución del gasto público durante septiembre de 2024.

El código fue desarrollado en **R** usando principalmente las librerías `tidyverse`, `dplyr` y `ggplot2`. En Google Colab esta parte puede presentarse como documentación en Markdown, ya que el entorno trabaja principalmente con Python, salvo que se configure un kernel adicional para R.

### ¿Qué hace este código?

1. **Limpia el entorno de trabajo**

   El script elimina objetos previos del entorno, limpia la consola y cierra gráficos anteriores. Esto permite iniciar la ejecución sin interferencias de procesos anteriores.

2. **Carga las librerías necesarias**

   Utiliza paquetes de R para manipulación y visualización de datos, principalmente:

   ```txt
   tidyverse
   dplyr
   ggplot2
   forcats
   pacman
   ```

3. **Crea la carpeta de salida**

   El código crea una carpeta llamada:

   ```txt
   outputs/graficos
   ```

   En esta carpeta se guardan las imágenes generadas durante la fase de visualización.

4. **Lee la base limpia**

   El script carga el archivo:

   ```txt
   gasto_publico_limpio_inferencia.csv
   ```

   Esta base proviene de la fase anterior de limpieza, transformación y validación de datos.

5. **Filtra registros válidos**

   Antes de graficar, el código elimina registros con valores faltantes en `monto_devengado` y `tasa_giro`. También conserva únicamente los registros donde `monto_devengado` es mayor que cero.

---

### Gráfico 1: Top 10 funciones con mayor gasto devengado

El primer gráfico agrupa los datos por la variable `funcion_nombre` y calcula la suma total de `monto_devengado` para cada función del gasto público. Luego selecciona las 10 funciones con mayor monto devengado y las representa mediante un gráfico de barras horizontales.

<p align="center">
  <img src="https://raw.githubusercontent.com/Rick2425/Trabajo-final-LP2-Practica/main/Rol_4_Visualizacion_de_resultados/grafico1_top_funciones.png"
       alt="Top 10 funciones con mayor gasto devengado"
       width="850">
</p>

Este gráfico permite identificar qué funciones del Estado concentran mayor gasto devengado durante septiembre de 2024. Se observa que las funciones con mayor ejecución se ubican en los primeros lugares del ranking, lo que ayuda a reconocer las áreas funcionales donde se concentró el gasto público durante el periodo analizado.

Archivo generado:

```txt
outputs/graficos/grafico1_top_funciones.png
```

---

### Gráfico 2: Distribución del monto devengado según categoría de gasto

El segundo gráfico compara la distribución del `monto_devengado` según la variable `categoria_gasto_nombre`. Para ello se utiliza un boxplot, el cual permite observar la mediana, dispersión y valores extremos del gasto según categoría.

Además, se aplica una escala logarítmica en el eje vertical para visualizar mejor las diferencias, debido a que los montos presupuestales pueden variar mucho entre registros.

<p align="center">
  <img src="https://raw.githubusercontent.com/Rick2425/Trabajo-final-LP2-Practica/main/Rol_4_Visualizacion_de_resultados/grafico2_boxplot_monto.png"
       alt="Distribución del monto devengado según categoría de gasto"
       width="850">
</p>

Este gráfico permite comparar el comportamiento del gasto corriente y el gasto de capital. La presencia de cajas, bigotes y valores extremos evidencia que el monto devengado no se distribuye de forma uniforme, sino que existen registros con montos considerablemente más altos que otros.

Archivo generado:

```txt
outputs/graficos/grafico2_boxplot_monto.png
```

---

### Gráfico 3: Top 10 regiones con mayor gasto devengado

El tercer gráfico agrupa los datos por `departamento_ejecutora_nombre` y calcula el total del `monto_devengado` por departamento. Luego selecciona las 10 regiones con mayor gasto devengado y las representa mediante un gráfico de barras.

<p align="center">
  <img src="https://raw.githubusercontent.com/Rick2425/Trabajo-final-LP2-Practica/main/Rol_4_Visualizacion_de_resultados/grafico3_top_regiones.png"
       alt="Top 10 regiones con mayor gasto devengado"
       width="850">
</p>

Este gráfico permite observar qué departamentos concentran mayores montos devengados durante el periodo analizado. La comparación regional ayuda a identificar diferencias territoriales en la ejecución presupuestal.

Archivo generado:

```txt
outputs/graficos/grafico3_top_regiones.png
```

---

### Archivos de salida

Como resultado de esta fase, se generan tres imágenes principales:

```
grafico1_top_funciones.png
grafico2_boxplot_monto.png
grafico3_top_regiones.png
---
### Importancia de esta fase

La visualización de datos permite interpretar de forma más clara los resultados obtenidos en las fases anteriores. Mientras que las tablas resumen muestran valores numéricos, los gráficos ayudan a identificar patrones, concentraciones y diferencias entre grupos.

En este proyecto, las visualizaciones permiten reconocer:

* Qué funciones concentran mayor gasto devengado.
* Cómo se distribuye el monto devengado según categoría de gasto.
* Qué regiones presentan mayor ejecución presupuestal.
* Qué diferencias existen entre los grupos analizados.

---

### Resultado de esta fase

En resumen, este bloque transforma la base limpia en resultados visuales. Los gráficos generados permiten comunicar los hallazgos principales del análisis de gasto público y sirven como evidencia del requisito de visualización solicitado en el proyecto integrador.

